In [14]:
import pandas as pd
import numpy as np
import json, warnings
warnings.filterwarnings("ignore")

panel = pd.read_parquet("data/panel.parquet")
funding_raw = pd.read_parquet("data/funding_rates_raw.parquet")
ohlcv = pd.read_parquet("data/ohlcv_daily.parquet")

with open("data/symbols.json") as f:
    all_symbols = json.load(f)

STUDY_START = panel["date"].min()
STUDY_END   = panel["date"].max()
OOS_START   = pd.Timestamp("2026-02-15")   # sealed OOS: Feb 15 – May 18, 2026

first_dates = panel.groupby("symbol")["date"].min().sort_values()
late_coins  = first_dates[first_dates > STUDY_START + pd.Timedelta(days=30)]

print(f"Study period: {STUDY_START.date()} to {STUDY_END.date()}")
print(f"IS window:    {STUDY_START.date()} to {(OOS_START - pd.Timedelta(days=1)).date()}")
print(f"OOS window:   {OOS_START.date()} to {STUDY_END.date()}")
print(f"\nCoins that listed mid-study (> 30 days after study start):")
print(late_coins.to_string())
print(f"\nSurvivorship note: {panel['symbol'].nunique()} coins have data.")
print("Coins delisted before the API pull date are NOT in this dataset — survivorship bias.")

Study period: 2025-05-18 to 2026-05-18
IS window:    2025-05-18 to 2026-02-14
OOS window:   2026-02-15 to 2026-05-18

Coins that listed mid-study (> 30 days after study start):
symbol
PUMP    2025-07-10
XPL     2025-08-22
WLFI    2025-08-23
ASTER   2025-09-19
MON     2025-10-08
LIT     2025-12-22
CHIP    2026-04-22

Survivorship note: 50 coins have data.
Coins delisted before the API pull date are NOT in this dataset — survivorship bias.


In [15]:
# --- Seal the OOS window ---
# IMPORTANT: OOS_START was already seen during the parameter sweep (sweep used all data).
# We therefore acknowledge the OOS is "technically burned" and treat this as a
# partial holdout. We report any OOS result honestly alongside this caveat.
# The IS-only sweep (Task 3) partially mitigates this by using only IS data to
# re-select parameters.

TOP_N      = 35
SYMBOLS    = all_symbols[:TOP_N]
SHORT_W    = 21   # parameters selected from sweep (acknowledged as mined)
LONG_W     = 90
HORIZON    = 14

panel_35   = panel[panel["symbol"].isin(SYMBOLS)].copy()
panel_is   = panel_35[panel_35["date"] < OOS_START].copy()
panel_oos  = panel_35[panel_35["date"] >= OOS_START].copy()

print(f"Top {TOP_N} coins in panel:   {panel_35['symbol'].nunique()}")
print(f"IS rows:   {len(panel_is):,}  ({panel_is['date'].nunique()} trading days)")
print(f"OOS rows:  {len(panel_oos):,} ({panel_oos['date'].nunique()} trading days)")

Top 35 coins in panel:   35
IS rows:   9,139  (273 trading days)
OOS rows:  3,255 (93 trading days)


In [16]:
import statsmodels.api as sm
from scipy import stats

def hac_tstat(series: pd.Series, horizon: int) -> tuple[float, float, int]:
    """
    HAC (Newey-West) t-statistic for H0: mean = 0.
    lag = max(horizon-1, NW-1994 auto bandwidth).
    Returns (t_stat, p_value, lags_used).
    """
    s = series.dropna().values
    T = len(s)
    auto_lag  = int(np.floor(4 * (T / 100) ** (2 / 9)))
    min_lag   = horizon - 1          # mechanical overlap from horizon-day returns
    max_lag   = max(auto_lag, min_lag)
    X = np.ones((T, 1))
    res = sm.OLS(s, X).fit(cov_type="HAC", cov_kwds={"maxlags": max_lag, "use_correction": True})
    return float(res.tvalues[0]), float(res.pvalues[0]), max_lag

In [17]:
from itertools import product as iproduct

short_windows   = [3, 7, 14, 21]
long_windows    = [14, 30, 60, 90]
forward_horizons = [1, 3, 5, 7, 14, 21]

def build_zscore(df: pd.DataFrame, sw: int, lw: int) -> pd.DataFrame:
    out = df.copy()
    grp = out.groupby("symbol")["daily_funding"]
    s_mean = grp.transform(lambda x: x.rolling(sw, min_periods=max(3, sw//2)).mean())
    l_mean = grp.transform(lambda x: x.rolling(lw, min_periods=max(10, lw//3)).mean())
    l_std  = grp.transform(lambda x: x.rolling(lw, min_periods=max(10, lw//3)).std())
    out["zscore"] = (s_mean - l_mean) / l_std
    out["zscore"] = out["zscore"].replace([np.inf, -np.inf], np.nan)
    return out

def daily_ic(df: pd.DataFrame, ret_col: str) -> pd.Series:
    sub = df.dropna(subset=["zscore", ret_col])
    def row_ic(g):
        return g["zscore"].corr(g[ret_col], method="spearman") if len(g) >= 5 else np.nan
    return sub.groupby("date").apply(row_ic).dropna()

# Build IS base (daily funding already in panel)
is_base = panel_is.copy()

is_results = []
for sw, lw in iproduct(short_windows, long_windows):
    if sw >= lw:
        continue
    df_z = build_zscore(is_base, sw, lw)
    for h in forward_horizons:
        ret_col = f"ret_{h}d"
        if ret_col not in df_z.columns:
            continue
        ic_series = daily_ic(df_z, ret_col)
        if len(ic_series) < 30:
            continue
        mean_ic = ic_series.mean()
        t, p, lags = hac_tstat(ic_series, horizon=h)
        is_results.append({
            "sw": sw, "lw": lw, "horizon": h,
            "mean_ic": mean_ic, "hac_t": t, "hac_p": p, "n_obs": len(ic_series)
        })
        
is_df = pd.DataFrame(is_results)
print(f"IS sweep: {len(is_df)} parameter combinations")
print("\nTop 10 by |HAC t-stat|:")
print(is_df.reindex(is_df["hac_t"].abs().sort_values(ascending=False).index)
      .head(10)[["sw","lw","horizon","mean_ic","hac_t","hac_p"]].to_string(index=False))

IS sweep: 56 parameter combinations

Top 10 by |HAC t-stat|:
 sw  lw  horizon   mean_ic     hac_t    hac_p
  7  30        1 -0.020033 -1.751182 0.079914
  3  30        1 -0.018604 -1.517798 0.129065
 21  30       21 -0.040367 -1.302462 0.192758
  7  14        1 -0.015401 -1.169411 0.242238
 21  30       14 -0.033073 -1.154703 0.248212
 14  30       21 -0.035181 -1.134064 0.256768
  3  14        1 -0.014813 -1.131755 0.257738
 21  90        7 -0.025173 -1.119859 0.262774
  3  60        1 -0.013034 -1.087415 0.276853
 21  30        1 -0.012390 -1.080991 0.279701


In [18]:
from statsmodels.stats.multitest import multipletests

p_values = is_df["hac_p"].values
N_tests  = len(p_values)

reject_bh,   p_bh,   _, _ = multipletests(p_values, alpha=0.05, method="fdr_bh")
reject_holm, p_holm, _, _ = multipletests(p_values, alpha=0.05, method="holm")

is_df["p_bh"]        = p_bh
is_df["p_holm"]      = p_holm
is_df["reject_bh"]   = reject_bh
is_df["reject_holm"] = reject_holm

n_sig_naive = (p_values < 0.05).sum()
n_sig_bh    = reject_bh.sum()
n_sig_holm  = reject_holm.sum()

print(f"Total IS combinations tested: {N_tests}")
print(f"Significant at naive p<0.05:  {n_sig_naive} ({n_sig_naive/N_tests:.0%})")
print(f"Significant after BH (FDR):   {n_sig_bh}   ({n_sig_bh/N_tests:.0%})")
print(f"Significant after Holm:       {n_sig_holm}  ({n_sig_holm/N_tests:.0%})")
print(f"\nNote: conventional t>2 threshold is far too lax for {N_tests} mined specs.")
print("The adjusted bar is roughly t>3 once the full search breadth is counted.")

# Show the best IS combo that survives BH correction
survivors = is_df[is_df["reject_bh"]].sort_values("mean_ic")
if len(survivors):
    print(f"\nBH-surviving combos (best IC first):")
    print(survivors[["sw","lw","horizon","mean_ic","hac_t","p_bh"]].head(10).to_string(index=False))
    BEST = survivors.iloc[0]
    BEST_SW, BEST_LW, BEST_H = int(BEST["sw"]), int(BEST["lw"]), int(BEST["horizon"])
    print(f"\nSelected for further tests: short={BEST_SW}d, long={BEST_LW}d, horizon={BEST_H}d")
else:
    print("\nNO combo survives BH correction — report as statistical null.")
    BEST_SW, BEST_LW, BEST_H = SHORT_W, LONG_W, HORIZON  # fall back to original for reporting

Total IS combinations tested: 56
Significant at naive p<0.05:  0 (0%)
Significant after BH (FDR):   0   (0%)
Significant after Holm:       0  (0%)

Note: conventional t>2 threshold is far too lax for 56 mined specs.
The adjusted bar is roughly t>3 once the full search breadth is counted.

NO combo survives BH correction — report as statistical null.
